# 00 · Setup and checks (shared)
**Owner:** Abdelrahman Elbehery · **Used by:** all four members

Run this notebook once when you start working in Colab, and again whenever `src/` changes:
1. connects Google Drive and gets the latest code from GitHub;
2. downloads SRSD Feynman Medium to the shared Drive folder (first time only);
3. runs the test suite on the real data;
4. prints the package versions to report in Section V.

**Before the first run**
- Runtime → Change runtime type → **T4 GPU**.
- If the repo is private: GitHub → Settings → Developer settings → Personal access tokens → create a token with *repo* read access. In Colab, open **Secrets** (🔑 on the left), add `GITHUB_TOKEN`, and turn on notebook access. Never paste the token into a cell.

## 1. Connect Drive and get the code

In [ ]:
# ---- Colab setup: run this cell first in every session ----
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO_URL = "https://github.com/Elbehery-237/ann-group13.git"   # change if the repo moves
REPO = "/content/ann-group13"

if os.path.exists(REPO):
    !git -C {REPO} pull -q                                   # new session in same VM: just update
else:
    try:                                                     # private repo: token stored in Colab Secrets
        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:                                        # public repo: no token needed
        url = REPO_URL
    !git clone -q {url} {REPO}
%cd {REPO}
!pip install -q -r requirements.txt

# Shared paths on Google Drive (data downloaded once, results survive disconnects)
DRIVE = "/content/drive/MyDrive/ann-group13"
os.environ["SRSD_ROOT"] = f"{DRIVE}/data/srsd-feynman_medium"
RESULTS, FIGURES = f"{DRIVE}/results", f"{DRIVE}/figures"
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| data found:", os.path.exists(os.environ["SRSD_ROOT"]))

## 2. Download SRSD Feynman Medium (first time only)
Downloads about 40 problems × 3 splits into the shared Drive folder. Skipped if the data is already there.
Share `MyDrive/ann-group13` with your teammates; each of them uses *Add shortcut to Drive* so the same path works for everyone.

In [ ]:
if not os.path.exists(os.environ["SRSD_ROOT"]):
    !git lfs install
    !git clone -q https://huggingface.co/datasets/yoshitomo-matsubara/srsd-feynman_medium {os.environ["SRSD_ROOT"]}
root = os.environ["SRSD_ROOT"]
n_eq = len([f for f in os.listdir(f"{root}/true_eq") if f.endswith(".pkl")])
print(n_eq, "equations")          # expected: 40
!head -n 2 {root}/train/feynman-i.39.10.txt

## 3. Run the tests
All tests should pass (≈ 3 minutes). If any fail, don't start training: fix or report first.

In [ ]:
!python -m pytest -q

## 4. Versions for Section V (Experimental setup)

In [ ]:
import platform, numpy, sympy
print("python ", platform.python_version())
print("torch  ", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("numpy  ", numpy.__version__)
print("sympy  ", sympy.__version__)
try:
    import kan; print("pykan  ", getattr(kan, "__version__", "installed"))
except ImportError:
    print("pykan   not installed")